In [ ]:
# !pip install numpy matplotlib opencv-python pillow torch torchvision
!pip install tqdm

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from PIL import Image
base_path = "/home/student/sky-scan/data"

# -----------------
# Check if CUDA is Available
# -----------------

# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"✅ Running on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected, running on CPU.")

# -----------------
# Define Dataset Paths
# -----------------
image_dir = f"{base_path}/patch"  # Folder with image tiles
mask_dir = f"{base_path}/patch-binary"  # Folder with corresponding mask tiles



# -----------------
# Function to Load Dataset
# -----------------
from tqdm import tqdm  # Import tqdm
import warnings
# Suppress all warnings
warnings.simplefilter('ignore', Image.DecompressionBombWarning)
# warnings.filterwarnings("ignore",message="libpng warning: iCCP: known incorrect sRGB profile")

def load_tiled_data(image_dir, mask_dir, image_size=(256, 256)):
    image_files = os.listdir(image_dir)
    num_images = len(image_files)

    # Preallocate memory for images and masks
    images = np.zeros((num_images, image_size[0], image_size[1], 3), dtype=np.float32)  # (num_images, height, width, channels)
    masks = np.zeros((num_images, image_size[0], image_size[1], 1), dtype=np.uint8)  # (num_images, height, width, channels)

    # Use tqdm to create a progress bar for the loop
    for idx, image_file in enumerate(tqdm(image_files, desc="Loading images", unit="image")):
        img_path = os.path.join(image_dir, image_file)
        mask_path = os.path.join(mask_dir, image_file)  # Mask should have the same filename

        if not os.path.exists(mask_path):
            continue  # Skip if corresponding mask is missing

        # Load and preprocess image
        image = Image.open(img_path).convert("RGB")  # Open image using PIL
        image = np.array(image) / 255.0 # Normalize

        # Load and preprocess mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, image_size)
        mask = (mask > 127).astype(np.uint8)  # Convert to binary
        mask = np.expand_dims(mask, axis=-1)  # Add channel dimension

        # Store the image and mask in the preallocated arrays
        images[idx] = image
        masks[idx] = mask

    return images, masks


# Load dataset
X, Y = load_tiled_data(image_dir, mask_dir)
print(f"✅ Loaded {len(X)} images and {len(Y)} masks.")

In [6]:


# !pip install torchsummary
# from torchsummary import summary
# -----------------
# Define Pix2Pix Generator in PyTorch
# -----------------
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()

        # Downsampling
        self.down1 = self.conv_block(3, 64)
        self.down2 = self.conv_block(64, 128)
        self.down3 = self.conv_block(128, 256)

        # Upsampling
        self.up1 = self.deconv_block(256, 128)
        self.up2 = self.deconv_block(128, 64)
        self.up3 = nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1)
        self.sigmoid = nn.Sigmoid()


    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2)
        )

    def deconv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self, x):
        x = self.down1(x)
        x = self.down2(x)
        x = self.down3(x)
        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        return self.sigmoid(x)

# Initialize the generator and move it to the GPU if available
generator = Generator().to(device)
# Print the summary of the generator model
# summary(generator, input_size=(3, 256, 256))

# -----------------
# Define Pix2Pix Discriminator in PyTorch
# -----------------
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        # Modify the input channels to 4 (image + mask)
        self.model = nn.Sequential(
            nn.Conv2d(4, 64, kernel_size=4, stride=2, padding=1),  # Change from 3 to 4 channels
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 1, kernel_size=4, stride=1, padding=1)
        )

    def forward(self, x):
        return self.model(x)

discriminator = Discriminator().to(device)
# summary(discriminator, input_size=(3, 256, 256))


# -----------------
# Define Loss & Optimizers
# -----------------
loss_fn = nn.BCEWithLogitsLoss()

# Optimizers
generator_optimizer = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
discriminator_optimizer = optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

def calculate_iou(pred_mask, true_mask):
    """
    Compute the Intersection over Union (IoU) score between predicted and true masks.

    :param pred_mask: Predicted binary mask (torch.Tensor or np.ndarray)
    :param true_mask: Ground truth binary mask (torch.Tensor or np.ndarray)
    :return: IoU score
    """
    pred_mask = pred_mask > 0.5  # Convert to binary (threshold at 0.5)
    true_mask = true_mask > 0.5  # Convert to binary

    intersection = (pred_mask & true_mask).sum().item()
    union = (pred_mask | true_mask).sum().item()

    iou = intersection / union if union > 0 else 1.0  # Avoid division by zero
    return iou

# -----------------
# Training Loop (Supports CUDA & CPU)
# -----------------
def train_step(input_image, target):
    # Convert images from (batch_size, height, width, channels) to (batch_size, channels, height, width)
    input_image = torch.tensor(input_image).to(device).permute(0, 3, 1, 2)  # Change shape to (B, C, H, W)
    target = torch.tensor(target).to(device).permute(0, 3, 1, 2)  # Change shape to (B, C, H, W)

    # Forward pass through generator and discriminator
    gen_output = generator(input_image)

    disc_real_output = discriminator(torch.cat((input_image, target), dim=1))
    disc_generated_output = discriminator(torch.cat((input_image, gen_output), dim=1))

    # Losses
    gen_loss = loss_fn(disc_generated_output, torch.ones_like(disc_generated_output)) + torch.mean(torch.abs(target - gen_output))
    disc_loss = (loss_fn(disc_real_output, torch.ones_like(disc_real_output)) +
                 loss_fn(disc_generated_output, torch.zeros_like(disc_generated_output)))

    # Backpropagation
    generator_optimizer.zero_grad()
    discriminator_optimizer.zero_grad()

    # Backprop through generator first
    gen_loss.backward(retain_graph=True)  # Retain the graph for the next backward pass

    # Backprop through discriminator
    disc_loss.backward()

    # Update the weights
    generator_optimizer.step()
    discriminator_optimizer.step()

    return gen_loss.item(), disc_loss.item()





EPOCHS = 60
BATCH_SIZE = 128

iou_scores = []  # Store IoU scores for visualization

for epoch in range(EPOCHS):
    progress_bar = tqdm(range(0, len(X), BATCH_SIZE), desc=f"Epoch {epoch+1}/{EPOCHS}", ncols=100)

    epoch_iou = 0  # Store average IoU for the epoch

    for i in progress_bar:
        batch_X = X[i:i+BATCH_SIZE]
        batch_Y = Y[i:i+BATCH_SIZE]

        gen_loss, disc_loss = train_step(batch_X, batch_Y)

        # Compute IoU for a small validation batch
        with torch.no_grad():
            input_tensor = torch.tensor(batch_X).to(device).permute(0, 3, 1, 2)  # Shape: (B, C, H, W)
            true_mask_tensor = torch.tensor(batch_Y).to(device).permute(0, 3, 1, 2)  # Shape: (B, C, H, W)

            pred_mask_tensor = generator(input_tensor)
            pred_mask_tensor = (pred_mask_tensor > 0.5).float()  # Binarize

            batch_iou = calculate_iou(pred_mask_tensor.cpu().numpy(), true_mask_tensor.cpu().numpy())
            epoch_iou += batch_iou

        progress_bar.set_postfix(Gen_Loss=gen_loss, Disc_Loss=disc_loss, IoU=batch_iou)

    epoch_iou /= len(X) // BATCH_SIZE  # Average IoU for the epoch
    iou_scores.append(epoch_iou)

    print(f"Epoch {epoch+1}/{EPOCHS} - Gen Loss: {gen_loss:.4f}, Disc Loss: {disc_loss:.4f}, IoU: {epoch_iou:.4f}")

plt.plot(range(1, EPOCHS + 1), iou_scores, marker='o', linestyle='-')
plt.xlabel("Epoch")
plt.ylabel("IoU Score")
plt.title("IoU Score Over Training Epochs")
plt.grid()
plt.show()



def visualize_results(input_img, true_mask, pred_mask):
    """
    Display input image, ground truth mask, predicted mask, and the difference.

    :param input_img: Original image (numpy array)
    :param true_mask: Ground truth binary mask (numpy array)
    :param pred_mask: Predicted binary mask (numpy array)
    """
    diff = np.abs(true_mask - pred_mask)  # Highlight differences

    fig, ax = plt.subplots(1, 4, figsize=(12, 4))
    ax[0].imshow(input_img)
    ax[0].set_title("Input Image")

    ax[1].imshow(true_mask.squeeze(), cmap="gray")
    ax[1].set_title("Ground Truth Mask")

    ax[2].imshow(pred_mask.squeeze(), cmap="gray")
    ax[2].set_title("Predicted Mask")

    ax[3].imshow(diff.squeeze(), cmap="jet")
    ax[3].set_title("Difference (Error)")

    for a in ax:
        a.axis("off")

    plt.show()

# Select a sample for visualization
idx = np.random.randint(len(X))
sample_input = X[idx]
sample_true_mask = Y[idx]

# Convert to tensor and predict
with torch.no_grad():
    input_tensor = torch.tensor(sample_input).to(device).permute(2, 0, 1).unsqueeze(0)  # Add batch dim
    pred_mask_tensor = generator(input_tensor)
    pred_mask = (pred_mask_tensor.cpu().numpy() > 0.5).astype(np.uint8).squeeze()  # Binarize

# Visualize
visualize_results(sample_input, sample_true_mask, pred_mask)

Epoch 1/60:   0%|                                                           | 0/127 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 512.00 MiB. GPU 0 has a total capacty of 15.77 GiB of which 147.25 MiB is free. Process 2582 has 14.63 GiB memory in use. Including non-PyTorch memory, this process has 872.00 MiB memory in use. Of the allocated memory 109.56 MiB is allocated by PyTorch, and 12.44 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF